<a href="https://colab.research.google.com/github/kinhalepuja31-cmd/Task24-Data-Quality-Audit/blob/main/Tasks_24.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np


In [3]:
# 1. LOAD DATA & STANDARDIZE TYPES

file_path = "/content/Global Superstore Data.xlsx"

orders_df = pd.read_excel(file_path, sheet_name="Orders")
returns_df = pd.read_excel(file_path, sheet_name="Returns")
people_df = pd.read_excel(file_path, sheet_name="People")

# Pre-convert dates to avoid inline syntax issues
orders_df['Order Date'] = pd.to_datetime(orders_df['Order Date'], errors='coerce')
orders_df['Ship Date'] = pd.to_datetime(orders_df['Ship Date'], errors='coerce')

# Clean string IDs and regions to ensure accurate relational matching
orders_df['Order ID'] = orders_df['Order ID'].astype(str).str.strip()
returns_df['Order ID'] = returns_df['Order ID'].astype(str).str.strip()
orders_df['Region'] = orders_df['Region'].astype(str).str.strip()
people_df['Region'] = people_df['Region'].astype(str).str.strip()

In [4]:
# 2. INITIALIZE AUTOMATED ISSUE LOG

issue_log = []

def log_issue(sheet, issue_type, description, count):
    """Helper function to format and record data anomalies."""
    if count > 0:
        issue_log.append({
            "Sheet": sheet,
            "Issue Type": issue_type,
            "Description": description,
            "Affected Rows": count
        })

print("\n--- Starting Data Quality Audit ---")



--- Starting Data Quality Audit ---


In [5]:
# 3. EXECUTE DATA QUALITY CHECKS

# Rule 1: Check for duplicate rows in Orders
duplicated_count = orders_df.duplicated().sum()
log_issue("Orders", "Duplicate Rows", "Identical rows found in dataset", duplicated_count)

# Rule 2: Check for missing values in critical columns
missing_columns = ['Order ID', 'Order Date', 'Customer ID', 'Sales', 'Quantity']
for col in missing_columns:
    missing_count = orders_df[col].isnull().sum()
    log_issue("Orders", "Missing Value", f"Null values in critical column: {col}", missing_count)

# Rule 3: Check for range violations (Quantity must be > 0)
invalid_qty_count = (orders_df['Quantity'] <= 0).sum()
log_issue("Orders", "Range Violation", "Quantity is 0 or negative", invalid_qty_count)

# Rule 4: Check for range violations (Sales must be positive)
invalid_sales_count = (orders_df['Sales'] <= 0).sum()
log_issue("Orders", "Range Violation", "Sales values are negative or zero", invalid_sales_count)

# Rule 5: Check for date consistency (Ship Date cannot be before Order Date)
invalid_dates_count = (orders_df['Ship Date'] < orders_df['Order Date']).sum()
log_issue("Orders", "Consistency Error", "Ship Date occurs before Order Date", invalid_dates_count)

# Rule 6: Relational Integrity (Returned Order IDs must exist in Master Orders sheet)
orphan_returns = returns_df[~returns_df['Order ID'].isin(orders_df['Order ID'])]
log_issue("Returns", "Relational Consistency", "Returned Order ID missing from Orders sheet", len(orphan_returns))

# Rule 7: Master Data Validation (Regions in Orders must exist in People master sheet)
invalid_regions = orders_df[~orders_df['Region'].isin(people_df['Region'])]
log_issue("Orders", "Master Data Error", "Region value does not match People sheet records", len(invalid_regions))


In [6]:
# 4. GENERATE DELIVERABLES

# Create the automated issue log DataFrame
issue_log_df = pd.DataFrame(issue_log)

# Print Summary Audit Report to Console
print("\n" + "="*30 + " AUDIT SUMMARY REPORT " + "="*30)
print(f"Total Sheets Audited: 3")
print(f"Total Unique Issues Found: {len(issue_log_df)}")
print("-" * 82)

if not issue_log_df.empty:
    print(issue_log_df.to_string(index=False))
else:
    print("DETAILED ISSUE LOG ---")
    print("Empty DataFrame\nColumns: [Sheet, Issue Type, Description, Affected Rows]\nIndex: []")
    print("\nNo anomalies detected. Your dataset is completely clean!")

print("-" * 82)

# Save deliverables to files
issue_log_df.to_csv("data_quality_issue_log.csv", index=False)

# Create a cleaned sample (removing duplicate rows if any exist)
cleaned_orders_df = orders_df.drop_duplicates()
cleaned_orders_df.head(100).to_csv("cleaned_sample_orders.csv", index=False)

print("\n... Audit Complete. Deliverables saved successfully! ...\n")


============================== AUDIT SUMMARY REPORT ==============================
Total Sheets Audited: 3
Total Unique Issues Found: 1
----------------------------------------------------------------------------------
 Sheet        Issue Type                                      Description  Affected Rows
Orders Master Data Error Region value does not match People sheet records            384
----------------------------------------------------------------------------------

... Audit Complete. Deliverables saved successfully! ...

